In [ ]:
#@title 1. Settings
PROJECT_DIR     = "" #@param {type:"string"}
#@markdown The project folder, used when it is not where DRIVE_ROOT and
#@markdown PROJECT_FOLDER below point. Set it to wherever you put this folder,
#@markdown on Drive or on the Colab disk, and every other path follows from it.
#@markdown Left empty it falls back to those two.
DRIVE_ROOT      = "/content/drive/MyDrive"  #@param {type:"string"}
PROJECT_FOLDER  = "KYNU truncated isoform" #@param {type:"string"}
BUILDS_ROOT     = "/content/drive/MyDrive/MDs/KYNU_truncated_isoform" #@param {type:"string"}
#@markdown Where the two CHARMM-GUI downloads were unpacked. The folder names in it
#@markdown are not typed here: a build folder is one holding step3_input.pdb, and its
#@markdown build is the word canonical or mixed in its path. The project's own md/ is
#@markdown searched too, and a root that does not exist is skipped.
NS_PER_BUILD    = 20     #@param {type:"integer"}
#@markdown How far to run now. The criterion is stated at 100 ns and is one-sided
#@markdown below it, so a first pass of 20 can stop the series and cannot pass it.
#@markdown Raise this to 100 if part 3 returns CONTINUE; finished chunks are skipped.
TOTAL_NS        = 100    #@param {type:"integer"}
#@markdown The planned total. The trajectory write interval is set from this and not
#@markdown from NS_PER_BUILD, so every chunk is sampled at the same spacing and a
#@markdown staged run does not end up denser at its start than at its end.
NS_PER_CHUNK    = 5      #@param {type:"integer"}
WALL_BUDGET_HOURS = 23.0 #@param {type:"number"}
#@markdown Stops cleanly here; Run all again to continue from the last chunk.
TARGET_FRAMES   = 2000   #@param {type:"integer"}
#@markdown Frames per build. The write interval is recomputed from this,
#@markdown since CHARMM-GUI's default is set for a 1 ns run.
BURN_FRACTION   = 0.2    #@param {type:"number"}
#@markdown Fraction of each trajectory discarded as settling before any number is taken.
CACHE_GROMACS   = True   #@param {type:"boolean"}
#@markdown Keep the compiled CUDA GROMACS on Drive so later sessions skip the build.
GMX_VER         = "2026.3"    #@param {type:"string"}
GMX_SIMD        = "AVX2_256"  #@param ["AVX2_256", "AVX_512", "SSE4.1", "AUTO"]
REF_T_EXPECTED  = 303.15 #@param {type:"number"}
#@markdown The thermostat temperature the builds must carry, used only when part 1's
#@markdown build spec predates its ref_t_K key; the spec's value wins when it is there.
MAX_COFACTOR_OFFSET_A = 3.0 #@param {type:"number"}
REPAIR_COFACTOR = True #@param {type:"boolean"}
#@markdown With the repair on, a cofactor built further than that from the pose it was
#@markdown given is moved back onto it before anything runs, and any solvent left
#@markdown inside that pose is moved to a position the ring vacated. Only coordinates
#@markdown change, so the atom count, index.ndx and topol.top stay as CHARMM-GUI wrote
#@markdown them, and the originals are kept beside them as .charmmgui_original. With
#@markdown it off, a badly built cofactor stops the run instead.
#@markdown CHARMM-GUI builds the substituted residue from the force field's internal
#@markdown coordinates, so its ring can come back in a default rotamer with the
#@markdown phosphate in solvent. The pose it built is compared against the file that
#@markdown was uploaded to it, and a cofactor further than this from where it was put
#@markdown stops the run before a chunk is spent on it.
MAXWARN         = 10     #@param {type:"integer"}

import os, shutil, time
T_START = time.time()
PROJECT = PROJECT_DIR.strip() or os.path.join(DRIVE_ROOT, PROJECT_FOLDER)
MD = os.path.join(PROJECT, "md")
BUILD = "mixed"
BUILD_ORDER = ["canonical", "mixed"]
assert BUILD in BUILD_ORDER
print("this notebook runs build:", BUILD.upper())
print("project      ", PROJECT)
if NS_PER_BUILD > TOTAL_NS:
    raise SystemExit("STOP: NS_PER_BUILD is %g and TOTAL_NS is %g. The write interval "
                     "comes from the planned total, so the total cannot be the "
                     "smaller of the two." % (NS_PER_BUILD, TOTAL_NS))
print("sampling     ", NS_PER_BUILD, "ns now, of a planned", TOTAL_NS, "ns")
print("wall budget  ", WALL_BUDGET_HOURS, "hours")


In [ ]:
#@title 2. Drive, GPU and GROMACS
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, glob, re, subprocess, shutil, datetime, math, collections
import numpy as np

PREFIX = "/usr/local/gromacs"
os.environ["PATH"] = PREFIX + "/bin:" + os.environ["PATH"]
RESULTS = os.path.join(MD, "results")
INPUTS = os.path.join(MD, "inputs")
DATA = os.path.join(PROJECT, "data")
os.makedirs(RESULTS, exist_ok=True)

def one_file(root, name, what):
    hits = []
    for dirpath, dirnames, filenames in os.walk(root):
        if name in filenames:
            hits.append(os.path.join(dirpath, name))
    if not hits:
        raise SystemExit("STOP: no %s anywhere under %s, which is where %s lives."
                         % (name, root, what))
    if len(hits) > 1:
        raise SystemExit("STOP: %d files named %s under %s, %s. Which one is %s is "
                         "not guessed here." % (len(hits), name, root, sorted(hits),
                                                what))
    return hits[0]


if not os.path.isdir(MD):
    raise SystemExit("STOP: no %s. Run part 1 first, and check PROJECT_DIR points at "
                     "the folder that holds it." % MD)
SPEC_PATH = one_file(MD, "build_spec.json", "the build spec part 1 writes")
SPEC = json.load(open(SPEC_PATH))
if sorted(set(BUILD_ORDER)) != sorted(set(SPEC["builds"])):
    raise SystemExit("STOP: part 1 wrote the builds %s and this notebook expects %s"
                     % (SPEC["builds"], BUILD_ORDER))
COFACTOR = SPEC["cofactor"]
RETAINED_END = SPEC["retained_end"]
LOST_FROM, LOST_TO = SPEC["lost_block"]
CHAIN_LENGTHS = SPEC["chain_lengths"]
COFACTOR_COPIES = SPEC["cofactor_copies"]
CRYSTAL = SPEC["crystal_partner_burial_A2"]
print("build spec: %s isoform %s, retained 1 to %d, lost %d to %d, cofactor %s"
      % (SPEC["accession"], SPEC["isoform"], RETAINED_END, LOST_FROM, LOST_TO,
         COFACTOR))
print("chain lengths expected: %s" % CHAIN_LENGTHS)
print("cofactor copies expected: %s" % COFACTOR_COPIES)
print("crystal partner burial of the cofactor, from the project's own measurement:")
for entry in sorted(CRYSTAL):
    print("   %-6s %6.2f A2, of which %6.2f A2 from residues %d to %d"
          % (entry, CRYSTAL[entry]["whole_partner"], CRYSTAL[entry]["lost_block"],
             LOST_FROM, LOST_TO))

REF_T_WANT = SPEC.get("ref_t_K")
REF_T_SOURCE = "part 1 build spec"
if REF_T_WANT is None:
    REF_T_WANT = REF_T_EXPECTED
    REF_T_SOURCE = "this notebook; the spec predates its ref_t_K key"
REF_T_WANT = float(REF_T_WANT)
print("thermostat: %g K expected, from %s" % (REF_T_WANT, REF_T_SOURCE))

SCHIFF_BASE_RESIDUE = SPEC.get("schiff_base_residue")
LYS_ATOMS = set(SPEC.get("lysine_atoms") or [])
BUILT_RESIDUE = list(SPEC.get("built_residue") or [])
MOIETY_HEAVY_MIN = SPEC.get("moiety_heavy_min")
if SCHIFF_BASE_RESIDUE is None or not LYS_ATOMS or not MOIETY_HEAVY_MIN:
    raise SystemExit("STOP: %s predates the internal aldimine: it carries no "
                     "schiff_base_residue, lysine_atoms or moiety_heavy_min. Run part "
                     "1 again, which measures them from the crystal structure, and "
                     "the cofactor will then be found by where it is bonded rather "
                     "than by a residue name that CHARMM may truncate." % SPEC_PATH)
UPLOADED = {}
if all(os.path.isfile((SPEC.get("inputs", {}).get(b) or {}).get("pdb") or "")
       for b in BUILD_ORDER):
    UPLOADED = {b: SPEC["inputs"][b]["pdb"] for b in BUILD_ORDER}
else:
    holding = [d for d, _dirs, files in os.walk(MD)
               if all("%s.pdb" % b in files for b in BUILD_ORDER)]
    if not holding:
        raise SystemExit("STOP: no folder under %s holds %s together. Part 1 writes "
                         "them side by side, and they carry the crystallographic "
                         "cofactor pose the built one is checked against."
                         % (MD, " and ".join("%s.pdb" % b for b in BUILD_ORDER)))
    if len(holding) > 1:
        raise SystemExit("STOP: %d folders under %s hold %s together, %s. Which set "
                         "part 1 wrote is not guessed here."
                         % (len(holding), MD,
                            " and ".join("%s.pdb" % b for b in BUILD_ORDER),
                            sorted(holding)))
    UPLOADED = {b: os.path.join(holding[0], "%s.pdb" % b) for b in BUILD_ORDER}
print("uploaded to CHARMM-GUI: %s"
      % ", ".join("%s %s" % (b, os.path.relpath(UPLOADED[b], MD))
                  for b in BUILD_ORDER))
print("cofactor: %s bonded to residue %d, %.2f A in the crystal; CHARMM-GUI builds "
      "it as %s" % (COFACTOR, SCHIFF_BASE_RESIDUE, SPEC.get("aldimine_A", float("nan")),
                    " or ".join(BUILT_RESIDUE) or "a substituted residue"))

CG_MARKER = "step3_input.pdb"
CG_REQUIRED = ["step3_input.gro", "step3_input.pdb", "step3_input.psf", "topol.top",
               "index.ndx", "step4.0_minimization.mdp", "step5_production.mdp"]


def find_run_dirs(roots, builds):
    found = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            if CG_MARKER in filenames:
                if dirpath not in [d for _, d, _ in found]:
                    found.append((root, dirpath,
                                  os.path.relpath(dirpath, root).lower()))
                dirnames[:] = []
    out = {}
    for b in builds:
        per_root = [(r, [d for rr, d, rel in found if rr == r and b in rel])
                    for r in roots]
        per_root = [(r, h) for r, h in per_root if h]
        if not per_root:
            raise SystemExit(
                "STOP: no folder holding a %s carries the word %s below any root "
                "searched, %s. Every folder holding a %s: %s."
                % (CG_MARKER, b, roots, CG_MARKER,
                   [d for _, d, _ in found] or "none"))
        root, hits = per_root[0]
        if len(hits) != 1:
            raise SystemExit(
                "STOP: %d folders under %s carry the word %s, %s. One per build is "
                "needed and this is not guessed." % (len(hits), root, b, sorted(hits)))
        out[b] = hits[0]
        for _, other in per_root[1:]:
            for o in other:
                for f in ("step3_input.pdb", "step3_input.gro"):
                    pa, po = os.path.join(out[b], f), os.path.join(o, f)
                    a = os.path.getsize(pa) if os.path.isfile(pa) else None
                    c = os.path.getsize(po) if os.path.isfile(po) else None
                    if a != c:
                        raise SystemExit(
                            "STOP: the %s build is in two places and the copies are "
                            "not the same system. %s has a %s of %s bytes and %s has "
                            "%s. Delete whichever is stale; which one runs is not "
                            "guessed." % (b, out[b], f, a, o, c))
                print("  note: %s is also at %s, same size, and that copy is not used"
                      % (b, o))
    return out


RUN_DIR = find_run_dirs([BUILDS_ROOT, MD], BUILD_ORDER)
print("\nCHARMM-GUI folders:")
for b in BUILD_ORDER:
    miss = [f for f in CG_REQUIRED if not os.path.isfile(os.path.join(RUN_DIR[b], f))]
    extra = "" if not miss else "   MISSING %s" % miss
    print("  %-10s %s%s" % (b, RUN_DIR[b], extra))
    if miss:
        raise SystemExit("STOP: %s is missing %s, so it is not a finished CHARMM-GUI "
                         "GROMACS folder. Unpack the download again and upload the "
                         "whole gromacs/ folder, toppar included." % (RUN_DIR[b], miss))
if not os.path.isdir(os.path.join(RUN_DIR[BUILD_ORDER[0]], "toppar")):
    raise SystemExit("STOP: no toppar/ beside %s. topol.top includes its .itp files by "
                     "relative path, so grompp fails without it."
                     % os.path.basename(RUN_DIR[BUILD_ORDER[0]]))

print("\n--- GPU ---")
HAVE_GPU = False
GPU_NAME = None
try:
    _g = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                         "--format=csv,noheader"], check=False, text=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if _g.returncode == 0 and (_g.stdout or "").strip():
        HAVE_GPU = True
        GPU_NAME = _g.stdout.strip().split("\n")[0]
        print(GPU_NAME)
        print("Chunk times scale with this card, so record it with the run: the same "
              "system takes several times longer on a T4 than on an A100.")
    else:
        print("nvidia-smi exited %d, so the runtime claims a GPU it cannot use."
              % _g.returncode)
        print((_g.stdout or "").strip()[:400])
except FileNotFoundError:
    print("no nvidia-smi, so this runtime has no GPU.")
    print("That is fine for the salvage and compare notebooks: they only call "
          "trjconv, trjcat, distance and rms, all of which run on the CPU.")
    print("A production run needs one. Runtime > Change runtime type > T4 GPU.")


def sh(cmd, cwd=None, label=""):
    print("\n$ " + cmd, flush=True)
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines = (r.stdout or "").split("\n")
    if len(lines) > 60:
        print("\n".join(lines[:25]))
        print("   ... %d lines omitted ..." % (len(lines) - 45))
        print("\n".join(lines[-20:]))
    else:
        print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError("%s FAILED (exit %d). The output above is the reason."
                           % (label or cmd, r.returncode))


def gmx_version():
    try:
        return subprocess.run(["gmx", "--version"], capture_output=True, text=True,
                              timeout=90).stdout
    except Exception:
        return ""


CACHE = os.path.join(PROJECT, "gromacs_cuda_%s.tar.gz" % GMX_VER)
CACHE_STALE = False

if "CUDA" in gmx_version():
    print("\nCUDA gmx already present in this session")
elif CACHE_GROMACS and os.path.isfile(CACHE):
    print("\nrestoring the cached GROMACS from", CACHE)
    try:
        sh("tar xzf '%s' -C /" % CACHE, label="restore cache")
    except Exception as e:
        print("the cache could not be read (%s); another session may be writing it. "
              "Building from source instead." % e)
        shutil.rmtree(PREFIX, ignore_errors=True)
    if "CUDA" not in gmx_version():
        print("the cached build does not run here, rebuilding from source")
        shutil.rmtree(PREFIX, ignore_errors=True)
        CACHE_STALE = True

if "CUDA" not in gmx_version():
    nvcc = subprocess.run("which nvcc", shell=True, capture_output=True,
                          text=True).stdout.strip()
    if not nvcc:
        raise SystemExit("STOP: nvcc not found, so there is no CUDA toolkit to build "
                         "against. Set the runtime to GPU and press Run all again.")
    print("\nbuilding GROMACS %s. 20 to 40 minutes, once per Drive folder." % GMX_VER)
    sh("apt-get -qq update && apt-get -qq install -y cmake build-essential "
       "libfftw3-dev", label="apt")
    if not os.path.isdir("/content/gromacs-%s" % GMX_VER):
        sh("wget -q https://ftp.gromacs.org/gromacs/gromacs-%s.tar.gz && "
           "tar xzf gromacs-%s.tar.gz" % (GMX_VER, GMX_VER), cwd="/content",
           label="download")
    os.makedirs("/content/gromacs-%s/build" % GMX_VER, exist_ok=True)
    sh("cmake .. -DGMX_BUILD_OWN_FFTW=OFF -DGMX_GPU=CUDA -DGMX_SIMD=%s "
       "-DCMAKE_INSTALL_PREFIX=%s -DGMX_DOUBLE=OFF -DBUILD_TESTING=OFF"
       % (GMX_SIMD, PREFIX), cwd="/content/gromacs-%s/build" % GMX_VER, label="cmake")
    sh("make -j$(nproc) && make install", cwd="/content/gromacs-%s/build" % GMX_VER,
       label="make")
    if "CUDA" not in gmx_version():
        raise SystemExit("STOP: the build finished but gmx reports no CUDA support.")
    if CACHE_GROMACS and (CACHE_STALE or not os.path.isfile(CACHE)):
        print("\ncaching the build to Drive so later sessions skip it")
        tmp = "%s.%d.tmp" % (CACHE, os.getpid())
        try:
            sh("tar czf '%s' %s" % (tmp, PREFIX.lstrip("/")), cwd="/", label="cache")
            os.replace(tmp, CACHE)
        except Exception as e:
            print("could not cache the build (%s); this costs the next session a "
                  "rebuild and nothing else." % e)
            if os.path.isfile(tmp):
                os.remove(tmp)
    elif CACHE_GROMACS:
        print("\nanother session already cached a build, leaving it alone")

print("\n" + gmx_version().split("\n")[0])


In [ ]:
#@title 3. Helpers
AA = set("ALA ARG ASN ASP CYS GLN GLU GLY HIS HSD HSE HSP ILE LEU LYS MET PHE PRO "
         "SER THR TRP TYR VAL".split())


def gmx(*args, label="", stdin_text=None, quiet=False, check=True):
    cmd = ["gmx"] + [str(a) for a in args]
    if not quiet:
        print("$ " + " ".join(cmd), flush=True)
    r = subprocess.run(cmd, text=True, input=stdin_text,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = r.stdout or ""
    lines = out.split("\n")
    if not quiet:
        if len(lines) > 80:
            print("\n".join(lines[:30]))
            print("   ... %d lines omitted ..." % (len(lines) - 60))
            print("\n".join(lines[-30:]))
        else:
            print(out)
    if r.returncode != 0 and check:
        if quiet:
            print(out[-3000:])
        raise RuntimeError("%s failed (exit %d). The output above is the reason."
                           % (label or cmd[1], r.returncode))
    return r.returncode, out


def must_exist(path, what):
    if not os.path.isfile(path):
        raise RuntimeError("%s reported success but %s was not written." % (what, path))


def read_pdb_atoms(path):
    out = []
    for L in open(path, errors="replace"):
        if not L.startswith(("ATOM", "HETATM")):
            continue
        seg = L[72:76].strip() or L[21].strip() or "?"
        out.append((int(L[6:11]), seg, int(L[22:26]), L[17:20].strip(),
                    L[12:16].strip()))
    return out


def protein_segments(atoms):
    rn = {}
    for ser, seg, ri, name, an in atoms:
        rn.setdefault((seg, ri), name)
    return sorted({seg for (seg, ri), name in rn.items() if name in AA})


def progress(**kw):
    p = os.path.join(RESULTS, "progress_%s.json" % BUILD)
    cur = json.load(open(p)) if os.path.isfile(p) else {}
    cur.update(kw)
    cur["updated"] = datetime.datetime.now().isoformat(timespec="seconds")
    cur["elapsed_hours"] = round((time.time() - T_START) / 3600.0, 2)
    json.dump(cur, open(p, "w"), indent=2)
    return cur


def budget_left():
    return WALL_BUDGET_HOURS - (time.time() - T_START) / 3600.0


def xtc_frames(path):
    if not os.path.isfile(path):
        return None
    r = subprocess.run([os.path.join(PREFIX, "bin", "gmx"), "check", "-f", path],
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = r.stdout or ""
    m = re.search(r"^\s*Coords\s+(\d+)", out, re.M)
    if m:
        return int(m.group(1))
    m = re.search(r"Last frame\s+(\d+)", out)
    if m:
        return int(m.group(1)) + 1
    if r.returncode != 0:
        return None
    raise SystemExit(
        "STOP: gmx check read %s and exited 0, but its output carried neither a "
        "Coords row nor a Last frame line, so the frame count is unknown. Fix "
        "this parse rather than letting every chunk be re-run as unsound. The "
        "first lines of the output were:\n%s"
        % (os.path.basename(path), "\n".join(out.split("\n")[:12])))


def xtc_last_time(path):
    if not os.path.isfile(path):
        return None
    r = subprocess.run([os.path.join(PREFIX, "bin", "gmx"), "check", "-f", path],
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    t = None
    for L in (r.stdout or "").split("\n"):
        m = re.search(r"frame\s+\d+\s+time\s+([-\d.eE+]+)", L)
        if m:
            t = float(m.group(1))
    return t


def want_frames(mdp, ns):
    dt = float(mdp_get(mdp, "dt"))
    every = int(mdp_get(mdp, "nstxout-compressed"))
    return int(round(ns * 1000.0 / dt / every))


def ndx_args(group="SOLU"):
    if not os.path.isfile("index.ndx"):
        raise SystemExit(
            "STOP: no index.ndx in %s, so the group %s cannot be resolved. It comes "
            "from the CHARMM-GUI download and is staged in with the other inputs, so "
            "if it is missing here the staging is wrong."
            % (os.getcwd(), group))
    return ("-n", "index.ndx")


def stage_in(run, work, n_chunks, per_chunk):
    os.makedirs(work, exist_ok=True)
    heavy = ("prod_all.xtc", "prod_nojump.xtc", "prod_fit.xtc")
    for name in sorted(os.listdir(run)):
        src = os.path.join(run, name)
        if os.path.isdir(src):
            if name == "toppar":
                shutil.copytree(src, os.path.join(work, name), dirs_exist_ok=True)
            continue
        if re.match(r"prod_\d\d\.xtc$", name) or name in heavy:
            continue
        shutil.copy2(src, os.path.join(work, name))
    kept = 0
    for i in range(1, n_chunks + 1):
        nm = "prod_%02d" % i
        src = os.path.join(run, nm + ".xtc")
        if not (os.path.isfile(src) and os.path.isfile(os.path.join(work, nm + ".gro"))):
            print("  %s absent, so it and every chunk after it is re-run" % nm)
            break
        shutil.copy2(src, os.path.join(work, nm + ".xtc"))
        got = xtc_frames(os.path.join(work, nm + ".xtc"))
        if got is None or got < per_chunk - 1:
            os.remove(os.path.join(work, nm + ".xtc"))
            print("  %s holds %s of %d frames, so it and every chunk after it is "
                  "re-run" % (nm, got, per_chunk))
            break
        kept += 1
    for i in range(kept + 1, n_chunks + 1):
        for ext in (".xtc", ".gro", ".cpt", "_prev.cpt", ".tpr", ".log", ".edr"):
            f = os.path.join(work, "prod_%02d%s" % (i, ext))
            if os.path.isfile(f):
                os.remove(f)
    print("  staged into %s, %d of %d chunks reusable" % (work, kept, n_chunks))
    return kept


DRIVE_PENDING = [0]
DRIVE_SENT = {}


def stage_out(work, run, names):
    for nm in names:
        src = os.path.join(work, nm)
        if not os.path.isfile(src):
            continue
        dst = os.path.join(run, nm)
        shutil.copy2(src, dst)
        got, want = os.path.getsize(dst), os.path.getsize(src)
        if got != want:
            raise SystemExit("STOP: %s is %d bytes in the Drive mount against %d on "
                             "disk, so even the cached copy is short."
                             % (nm, got, want))
        DRIVE_PENDING[0] += want
        DRIVE_SENT[dst] = want


def drive_wait(why="", floor=20.0, mbps=12.0):
    mb = DRIVE_PENDING[0] / 1e6
    DRIVE_PENDING[0] = 0
    if mb <= 0:
        return
    secs = max(floor, mb / mbps)
    print("  waiting %.0f s for Drive to take %.0f MB%s"
          % (secs, mb, " after " + why if why else ""), flush=True)
    os.sync()
    time.sleep(secs)


def drive_verify():
    try:
        from google.colab import drive as _drive
        os.sync()
        print("\nflushing the Drive mount, then checking what Drive actually holds")
        _drive.flush_and_unmount()
        _drive.mount("/content/drive")
    except Exception as _e:
        print("\ncould not flush and remount Drive (%s), so the uploads are "
              "unverified. Leave this tab open for a couple of minutes before "
              "closing it." % _e)
        return
    bad, n = [], 0
    for path, want in sorted(DRIVE_SENT.items()):
        n += 1
        got = os.path.getsize(path) if os.path.isfile(path) else None
        if got != want:
            bad.append((os.path.basename(path), got, want))
    if bad:
        for nm, got, want in bad:
            print("   %-22s %s bytes on Drive against %d written" % (nm, got, want))
        raise SystemExit(
            "STOP: %d of %d files are short or missing on Drive after the flush, so "
            "the upload did not finish. The frames are not lost: open this notebook "
            "again and press Run all, which re-sends what is short and skips every "
            "chunk that is whole." % (len(bad), n))
    print("Drive holds all %d uploaded files at full size. Safe to close the runtime."
          % n)


print("helpers ready, MAXWARN =", MAXWARN)


In [ ]:
#@title 4. Run the build. This is the long cell.
AA_PLUS = AA | set(BUILT_RESIDUE)
REPAIRED = {}


def read_pdb_full(path):
    out = []
    for L in open(path, errors="replace"):
        if not L.startswith(("ATOM", "HETATM")):
            continue
        seg = L[72:76].strip() or L[21].strip() or "?"
        out.append((int(L[6:11]), seg, int(L[22:26]), L[17:21].strip(),
                    L[12:16].strip(),
                    (float(L[30:38]), float(L[38:46]), float(L[46:54]))))
    return out


def read_pdb_atoms_wide(path):
    return [t[:5] for t in read_pdb_full(path)]


def gro_residues(path):
    out = []
    with open(path, errors="replace") as fh:
        fh.readline()
        n = int(fh.readline())
        prev = None
        for _ in range(n):
            L = fh.readline()
            key = (L[:5].strip(), L[5:10].strip())
            if key != prev:
                out.append(key[1])
                prev = key
    return out


def chain_residues(atoms):
    out = collections.defaultdict(set)
    for ser, seg, ri, name, an in atoms:
        if name in AA_PLUS:
            out[seg].add(ri)
    return {k: sorted(v) for k, v in out.items()}


def protein_segments_k(atoms):
    rn = {}
    for ser, seg, ri, name, an in atoms:
        rn.setdefault((seg, ri), name)
    return sorted({seg for (seg, ri), name in rn.items() if name in AA_PLUS})


def moiety_atoms(atoms, seg):
    return {an for ser, sg, ri, name, an in atoms
            if sg == seg and ri == SCHIFF_BASE_RESIDUE and an not in LYS_ATOMS}


def cofactor_residues(atoms):
    segs = {seg for ser, seg, ri, name, an in atoms if ri == SCHIFF_BASE_RESIDUE}
    out = []
    for seg in segs:
        heavy = {a for a in moiety_atoms(atoms, seg) if not a.startswith("H")}
        if len(heavy) >= MOIETY_HEAVY_MIN:
            out.append((seg, SCHIFF_BASE_RESIDUE))
    return sorted(out)


def kabsch(P, Q):
    P, Q = np.asarray(P, dtype=float), np.asarray(Q, dtype=float)
    cp, cq = P.mean(0), Q.mean(0)
    U, S, Vt = np.linalg.svd((P - cp).T @ (Q - cq))
    d = 1.0 if np.linalg.det(Vt.T @ U.T) > 0 else -1.0
    R = Vt.T @ np.diag([1.0, 1.0, d]) @ U.T
    moved = (R @ (P - cp).T).T + cq
    return R, cp, cq, float(np.sqrt(((moved - Q) ** 2).sum(1).mean()))


COFACTOR_ATOM_MAP = {"C4'": "C4A", "C4": "C4", "C3": "C3", "O3": "O3", "C2": "C2",
                     "C2'": "C2A", "N1": "N1", "C6": "C6", "C5": "C5", "C5'": "C5A",
                     "OP4": "O4P", "P": "P", "OP1": "O1P", "OP2": "O2P", "OP3": "O3P"}
MAX_BOND_MISMATCH_A = 0.25
CLASH_A = 2.6
MIN_CONTACT_A = 1.2
SOLVENT_NAMES = ("TIP3", "HOH", "SOD", "POT", "CLA", "MG", "ZN", "CAL")


def cofactor_targets(run, build):
    built = read_pdb_full(os.path.join(run, "step3_input.pdb"))
    given = read_pdb_full(UPLOADED[build])
    short = [t[:5] for t in built]
    built_hosts = [seg for seg, ri in cofactor_residues(short)]
    given_hosts = []
    for ser, seg, ri, name, an, xyz in given:
        if name == COFACTOR and seg not in given_hosts:
            given_hosts.append(seg)
    if len(built_hosts) != len(given_hosts):
        raise SystemExit("STOP: %s was uploaded with %d copies of %s and came back "
                         "with %d substituted residues. Every copy that went in has "
                         "to come back, or the pocket measured is not the one that "
                         "was set up." % (build, len(given_hosts), COFACTOR,
                                          len(built_hosts)))
    order = []
    for ser, seg, ri, name, an, xyz in built:
        if seg in built_hosts and seg not in order:
            order.append(seg)
    out = {}
    for b_seg, g_seg in zip(order, given_hosts):
        b_ca = {ri: xyz for ser, seg, ri, name, an, xyz in built
                if seg == b_seg and an == "CA" and ri <= RETAINED_END}
        g_ca = {ri: xyz for ser, seg, ri, name, an, xyz in given
                if seg == g_seg and an == "CA" and ri <= RETAINED_END}
        shared = sorted(set(b_ca) & set(g_ca))
        if len(shared) < 50:
            raise SystemExit("STOP: the uploaded chain %s and the built chain %s share "
                             "%d alpha carbons at or below residue %d, too few to "
                             "superpose on" % (g_seg, b_seg, len(shared), RETAINED_END))
        R, cp, cq, rmsd = kabsch([g_ca[i] for i in shared], [b_ca[i] for i in shared])
        if rmsd > 1.0:
            raise SystemExit("STOP: the chain that went into CHARMM-GUI fits the one "
                             "that came back at %.2f A over %d alpha carbons. Adding "
                             "hydrogens and substituting one residue does not move a "
                             "backbone, so these are not the same structure."
                             % (rmsd, len(shared)))
        g_cof = {an: xyz for ser, seg, ri, name, an, xyz in given
                 if seg == g_seg and name == COFACTOR}
        b_cof = {an: xyz for ser, seg, ri, name, an, xyz in built
                 if seg == b_seg and ri == SCHIFF_BASE_RESIDUE
                 and an in COFACTOR_ATOM_MAP}
        missing = ([a for a in COFACTOR_ATOM_MAP if a not in b_cof]
                   + [a for a in COFACTOR_ATOM_MAP.values() if a not in g_cof])
        if missing:
            raise SystemExit("STOP: the cofactor atom names do not match what this "
                             "notebook knows. Missing %s. Built atoms on %s: %s. "
                             "Uploaded atoms on %s: %s."
                             % (sorted(set(missing)), b_seg,
                                sorted(moiety_atoms(short, b_seg)), g_seg,
                                sorted(g_cof)))
        names = sorted(COFACTOR_ATOM_MAP)
        A = np.asarray([b_cof[n] for n in names], dtype=float)
        B = np.asarray([g_cof[COFACTOR_ATOM_MAP[n]] for n in names], dtype=float)
        da = np.linalg.norm(A[:, None] - A[None], axis=-1)
        db = np.linalg.norm(B[:, None] - B[None], axis=-1)
        bonded = (da < 1.8) & (da > 0.1)
        worst_bond = float(np.abs(da - db)[bonded].max()) if bonded.any() else 99.0
        if worst_bond > MAX_BOND_MISMATCH_A:
            raise SystemExit("STOP: under the atom name map the two cofactors disagree "
                             "by %.2f A on a bonded distance, so the map is pairing "
                             "the wrong atoms. The same molecule in two conformations "
                             "keeps its bond lengths." % worst_bond)
        target = {n: (R @ (np.asarray(g_cof[COFACTOR_ATOM_MAP[n]], dtype=float) - cp))
                     + cq for n in names}
        d_p = float(np.linalg.norm(target["P"] - np.asarray(b_cof["P"], dtype=float)))
        d_mid = float(np.linalg.norm(
            np.mean([target[n] for n in names], axis=0) - A.mean(0)))
        out[b_seg] = dict(target=target, built={n: tuple(b_cof[n]) for n in names},
                          offset_P=d_p, offset_centroid=d_mid, backbone_rmsd=rmsd,
                          alpha_carbons=len(shared), bond_mismatch=worst_bond)
    return built, out


def cofactor_pose(run, build, label="cofactor"):
    built, info = cofactor_targets(run, build)
    worst = 0.0
    for seg in sorted(info):
        d = info[seg]
        worst = max(worst, d["offset_P"], d["offset_centroid"])
        print("  %s on %s: backbone fit %.2f A over %d alpha carbons, phosphorus "
              "%.2f A and centroid %.2f A from the uploaded pose"
              % (label, seg, d["backbone_rmsd"], d["alpha_carbons"], d["offset_P"],
                 d["offset_centroid"]))
    return worst


def bond_graph(atoms):
    names = sorted(atoms)
    g = {n: set() for n in names}
    heavy = [n for n in names if not n.startswith("H")]
    for i, a in enumerate(heavy):
        for b in heavy[i + 1:]:
            if np.linalg.norm(atoms[a] - atoms[b]) < 1.95:
                g[a].add(b)
                g[b].add(a)
    for h in names:
        if not h.startswith("H"):
            continue
        near = min(heavy, key=lambda n: np.linalg.norm(atoms[h] - atoms[n]))
        if np.linalg.norm(atoms[h] - atoms[near]) < 1.35:
            g[h].add(near)
            g[near].add(h)
    return g


def beyond(g, root, through):
    seen, stack = {through}, [through]
    while stack:
        n = stack.pop()
        for m in g[n]:
            if n == through and m == root:
                continue
            if m not in seen:
                seen.add(m)
                stack.append(m)
    return None if root in seen else seen


def best_torsion(points, targets, origin, axis):
    u = axis / np.linalg.norm(axis)
    a = points - origin
    b = targets - origin
    par = np.outer(a @ u, u)
    perp = a - par
    A = float((perp * b).sum())
    B = float((np.cross(u, perp) * b).sum())
    return float(np.arctan2(B, A))


def rotate_about(points, origin, axis, angle):
    u = axis / np.linalg.norm(axis)
    c, s = np.cos(angle), np.sin(angle)
    a = np.asarray(points, dtype=float) - origin
    return origin + a * c + np.cross(u, a) * s + np.outer(a @ u, u) * (1.0 - c)


def fit_cofactor_torsions(coords, target, anchor, sweeps=12, starts=36):
    atoms = {k: np.asarray(v, dtype=float) for k, v in coords.items()}
    g = bond_graph(atoms)
    rotatable = []
    for a in sorted(atoms):
        if a.startswith("H"):
            continue
        for b in sorted(g[a]):
            if b.startswith("H"):
                continue
            side = beyond(g, a, b)
            if side is None:
                continue
            if any(n in anchor and n != b for n in side):
                continue
            if not any(n in target and n != b for n in side):
                continue
            rotatable.append((a, b, sorted(side)))
    rotatable.sort(key=lambda t: len(t[2]), reverse=True)
    scored = sorted(target)

    def score(st):
        return float(np.sqrt(np.mean([np.sum((st[n] - target[n]) ** 2)
                                      for n in scored])))

    def descend(st):
        log = {}
        for _ in range(sweeps):
            before = score(st)
            for a, b, side in rotatable:
                moving = [n for n in side if n in target and n != b]
                if not moving:
                    continue
                ang = best_torsion(np.asarray([st[n] for n in moving]),
                                   np.asarray([target[n] for n in moving]),
                                   st[a], st[b] - st[a])
                new = rotate_about([st[n] for n in side], st[a], st[b] - st[a], ang)
                for n, p in zip(side, new):
                    st[n] = p
                log[(a, b)] = log.get((a, b), 0.0) + np.degrees(ang)
            if before - score(st) < 1e-4:
                break
        return st, score(st), log

    best = None
    if rotatable:
        a0, b0, side0 = rotatable[0]
        for k in range(max(1, starts)):
            st = {n: p.copy() for n, p in atoms.items()}
            ang0 = 2.0 * np.pi * k / max(1, starts)
            new = rotate_about([st[n] for n in side0], st[a0], st[b0] - st[a0], ang0)
            for n, p in zip(side0, new):
                st[n] = p
            st, rms, log = descend(st)
            log[(a0, b0)] = log.get((a0, b0), 0.0) + np.degrees(ang0)
            if best is None or rms < best[1]:
                best = (st, rms, log)
    else:
        best = descend({n: p.copy() for n, p in atoms.items()})
    return best


def contact_check(atoms, info):
    cof = np.asarray([t[5] for t in atoms
                      if t[1] in info and t[2] == SCHIFF_BASE_RESIDUE
                      and t[4] not in LYS_ATOMS], dtype=float)
    cof_heavy = np.asarray([t[5] for t in atoms
                            if t[1] in info and t[2] == SCHIFF_BASE_RESIDUE
                            and t[4] not in LYS_ATOMS
                            and not t[4].startswith("H")], dtype=float)
    rest, rest_heavy = [], []
    for t in atoms:
        if t[1] in info and t[2] == SCHIFF_BASE_RESIDUE:
            continue
        rest.append(t[5])
        if not t[4].startswith("H"):
            rest_heavy.append(t[5])
    rest = np.asarray(rest, dtype=float)
    rest_heavy = np.asarray(rest_heavy, dtype=float)
    hh = min(float(np.linalg.norm(rest_heavy - p, axis=1).min()) for p in cof_heavy)
    any_ = min(float(np.linalg.norm(rest - p, axis=1).min()) for p in cof)
    return hh, any_


def repair_cofactor(run, build):
    gro = os.path.join(run, "step3_input.gro")
    pdb = os.path.join(run, "step3_input.pdb")
    for path in (gro, pdb):
        keep = path + ".charmmgui_original"
        if not os.path.isfile(keep):
            shutil.copy2(path, keep)
    built, info = cofactor_targets(run, build)

    moved, vacated, pocket = {}, [], []
    where = {}
    for i, (ser, sg, ri, name, an, xyz) in enumerate(built):
        where[(sg, ri, an)] = i
    for seg in sorted(info):
        d = info[seg]
        whole = {an: np.asarray(xyz, dtype=float)
                 for ser, sg, ri, name, an, xyz in built
                 if sg == seg and ri == SCHIFF_BASE_RESIDUE}
        anchor = [a for a in whole if a in LYS_ATOMS]
        fitted, rms, turned = fit_cofactor_torsions(whole, d["target"], set(anchor))
        for a in anchor:
            if float(np.linalg.norm(fitted[a] - whole[a])) > 1e-6:
                raise SystemExit("STOP: the repair moved %s, which belongs to the "
                                 "lysine and has to stay where CHARMM-GUI put it"
                                 % a)
        print("  %s: cofactor now %.2f A from the crystallographic pose, by turning "
              "%s" % (seg, rms, ", ".join(
                  "%s-%s by %.0f degrees" % (a, b, abs((t + 180) % 360 - 180))
                  for (a, b), t in sorted(turned.items(),
                                          key=lambda kv: -abs(kv[1])))))
        for a in whole:
            if a in LYS_ATOMS:
                continue
            moved[where[(seg, SCHIFF_BASE_RESIDUE, a)]] = fitted[a]
            if not a.startswith("H"):
                vacated.append(whole[a])
                pocket.append(fitted[a])
    pocket = np.asarray(pocket)

    inside = {}
    for i, (ser, seg, ri, name, an, xyz) in enumerate(built):
        if name in SOLVENT_NAMES:
            if float(np.linalg.norm(pocket - np.asarray(xyz), axis=1).min()) < CLASH_A:
                inside.setdefault((seg, ri, name), []).append((i, xyz))
    free = sorted(vacated,
                  key=lambda p: -float(np.linalg.norm(pocket - p, axis=1).min()))
    if len(inside) > len(free):
        raise SystemExit("STOP: %d solvent molecules sit inside the corrected pose and "
                         "only %d positions are vacated by turning the cofactor, so "
                         "there is nowhere to put them without changing what the "
                         "system is made of." % (len(inside), len(free)))
    for k, (key, atoms) in enumerate(sorted(inside.items())):
        atoms.sort()
        shift = free[k] - np.asarray(atoms[0][1], dtype=float)
        for i, xyz in atoms:
            moved[i] = np.asarray(xyz, dtype=float) + shift
    print("  repair: %d cofactor copies turned into the pocket, %d solvent molecules "
          "moved to positions the ring vacated" % (len(info), len(inside)))

    out, n, i = [], 0, 0
    for L in open(pdb, errors="replace"):
        if not L.startswith(("ATOM", "HETATM")):
            out.append(L)
            continue
        if i in moved:
            x, y, z = moved[i]
            L = "%s%8.3f%8.3f%8.3f%s" % (L[:30], x, y, z, L[54:])
            n += 1
        i += 1
        out.append(L)
    if n != len(moved):
        raise SystemExit("STOP: %d atoms were to be moved and %d lines of %s matched. "
                         "A mismatch here would leave part of the cofactor behind."
                         % (len(moved), n, os.path.basename(pdb)))
    open(pdb, "w").writelines(out)

    lines = open(gro, errors="replace").read().splitlines(True)
    count = int(lines[1])
    body, n = lines[2:2 + count], 0
    if count != len(built):
        raise SystemExit("STOP: %s holds %d atoms and %s holds %d, so they are not "
                         "the same system" % (os.path.basename(gro), count,
                                              os.path.basename(pdb), len(built)))
    for i in range(count):
        if i in moved:
            x, y, z = moved[i]
            body[i] = "%s%8.3f%8.3f%8.3f\n" % (body[i][:20], x / 10.0, y / 10.0,
                                               z / 10.0)
            n += 1
    if n != len(moved):
        raise SystemExit("STOP: %d atoms were to be moved and %d lines of %s matched, "
                         "so the .gro and the .pdb do not share an atom order."
                         % (len(moved), n, os.path.basename(gro)))
    open(gro, "w").writelines(lines[:2] + body + lines[2 + count:])

    was = contact_check(built, info)
    now = contact_check(read_pdb_full(pdb), info)
    print("  repair: closest contact between a cofactor atom and another residue, "
          "%.2f A heavy to heavy (%.2f A counting hydrogens), against %.2f A (%.2f A) "
          "in the file CHARMM-GUI delivered"
          % (now[0], now[1], was[0], was[1]))
    if now[0] < was[0] or now[1] < was[1]:
        raise SystemExit("STOP: the repair leaves the cofactor in tighter contact "
                         "with the rest of the system than CHARMM-GUI's own file did, "
                         "so it has made the structure worse rather than better.")
    if now[0] < MIN_CONTACT_A:
        raise SystemExit("STOP: after the repair a cofactor heavy atom sits %.2f A "
                         "from an atom of another residue, below the %.1f A floor. A "
                         "minimisation cannot open a contact shorter than a bond."
                         % (now[0], MIN_CONTACT_A))
    return len(inside), now[0]


def check_identity(run, build):
    PDB = read_pdb_atoms_wide(os.path.join(run, "step3_input.pdb"))
    gro = os.path.join(run, "step3_input.gro")
    with open(gro, errors="replace") as fh:
        fh.readline()
        n_gro = int(fh.readline())
    if len(PDB) != n_gro:
        raise SystemExit("STOP: %s step3_input.pdb has %d atoms and the .gro has %d. "
                         "They must be the same build." % (build, len(PDB), n_gro))
    prot = protein_segments_k(PDB)
    if len(prot) != 2:
        raise SystemExit("STOP: %s has %d protein segments (%s). This measures one "
                         "interface between two subunits, so a monomer or a trimer "
                         "answers a different question."
                         % (build, len(prot), ",".join(prot)))
    res = chain_residues(PDB)
    got = sorted(len(res[s]) for s in prot)
    want = sorted(CHAIN_LENGTHS[build])
    if got != want:
        raise SystemExit("STOP: %s carries chains of %s residues and the build spec "
                         "says %s. Either the wrong model was uploaded to CHARMM-GUI "
                         "or the two downloads went in the wrong order. Which one "
                         "this is is not guessed." % (build, got, want))
    cof = cofactor_residues(PDB)
    if len(cof) != COFACTOR_COPIES[build]:
        raise SystemExit("STOP: %s carries %d copies of %s and the spec says %d. The "
                         "pocket measurement is about the cofactor, so a build with "
                         "the wrong number of them measures something else."
                         % (build, len(cof), COFACTOR, COFACTOR_COPIES[build]))
    named = collections.Counter(n for n in gro_residues(gro) if n in BUILT_RESIDUE)
    if sum(named.values()) != COFACTOR_COPIES[build]:
        raise SystemExit("STOP: %s holds %d residue named %s and the spec says %d "
                         "copies of the cofactor. The substitution on CHARMM-GUI's "
                         "PDB manipulation page is what puts them there."
                         % (gro, sum(named.values()), " or ".join(BUILT_RESIDUE),
                            COFACTOR_COPIES[build]))
    if len(named) > 1:
        raise SystemExit("STOP: %s mixes %s. The two differ by a proton on the "
                         "phosphate, so a system carrying one of each is not one "
                         "model of the cofactor."
                         % (gro, " and ".join("%d %s" % (v, k)
                                              for k, v in sorted(named.items()))))
    built_as = list(named)[0]
    if built_as != BUILT_RESIDUE[0]:
        print("  note: built as %s, not %s. The two differ by one proton on the "
              "phosphate. Both builds have to carry the same one, which the other "
              "notebook's own check repeats." % (built_as, BUILT_RESIDUE[0]))
    hosts = sorted({seg for seg, ri in cof})
    print("  identity ok: %s, chains of %s residues on %s, %d copies of %s as %s%d "
          "on %s, %d atoms" % (build, got, ",".join(prot), len(cof), COFACTOR,
                               built_as, SCHIFF_BASE_RESIDUE, ",".join(hosts),
                               len(PDB)))
    worst = cofactor_pose(run, build)
    if worst > MAX_COFACTOR_OFFSET_A:
        if not REPAIR_COFACTOR:
            raise SystemExit(
                "STOP: CHARMM-GUI built the cofactor %.2f A from where it was handed "
                "to it, past the %.1f A allowed, and the repair is switched off. It "
                "builds a substituted residue from internal coordinates, so the ring "
                "has come back in a default rotamer with the phosphate out of its "
                "cup, and running from here would measure a cofactor leaving a pocket "
                "it was never in." % (worst, MAX_COFACTOR_OFFSET_A))
        print("  CHARMM-GUI built the cofactor %.2f A from the pose it was given, "
              "past the %.1f A allowed, which is the default rotamer its internal "
              "coordinate build produces. Putting it back."
              % (worst, MAX_COFACTOR_OFFSET_A))
        relocated, closest = repair_cofactor(run, build)
        worst = cofactor_pose(run, build, label="cofactor after repair")
        if worst > MAX_COFACTOR_OFFSET_A:
            raise SystemExit("STOP: the cofactor is still %.2f A from the "
                             "crystallographic pose after the repair, so the repair "
                             "did not take." % worst)
        REPAIRED[build] = dict(relocated_solvent=relocated,
                               closest_contact_A=round(closest, 2),
                               offset_after_A=round(worst, 2))
    return len(PDB), prot


def interface_groups(build):
    REF = read_pdb_atoms_wide("prod_ref.pdb")
    prot = protein_segments_k(REF)
    if len(prot) != 2:
        raise SystemExit("STOP: prod_ref.pdb has %d protein segments, expected 2"
                         % len(prot))
    BY, SEG, COF = {}, collections.defaultdict(list), collections.defaultdict(list)
    res_of = {}
    drift = 0
    for i, (ser, seg, ri, name, an) in enumerate(REF, 1):
        if ser != i:
            drift += 1
        BY[(seg, ri, an)] = i
        res_of.setdefault((seg, ri), name)
        if name in AA_PLUS:
            SEG[seg].append(i)
        if ri == SCHIFF_BASE_RESIDUE and an not in LYS_ATOMS:
            COF[(seg, ri)].append(i)
    if drift:
        print("  note: %d of %d atoms carry a serial that differs from their position "
              "in the file; the index groups use the position, which is what gmx reads"
              % (drift, len(REF)))
    if len(COF) != COFACTOR_COPIES[build]:
        raise SystemExit("STOP: prod_ref.pdb holds %d copies of %s and the spec says "
                         "%d, so the trajectory is not the system that was checked"
                         % (len(COF), COFACTOR, COFACTOR_COPIES[build]))

    lengths = {s: len({ri for (sg, ri) in res_of
                       if sg == s and res_of[(sg, ri)] in AA_PLUS})
               for s in prot}
    hosts = sorted({seg for seg, ri in COF})
    by_length = sorted(prot, key=lambda s: -lengths[s])
    canonical_seg = by_length[0]
    if canonical_seg not in hosts:
        raise SystemExit("STOP: the longer chain is %s with %d residues and the "
                         "cofactor sits on %s. The pocket scored here is the one in "
                         "the canonical subunit, and these two do not agree."
                         % (canonical_seg, lengths[canonical_seg], ",".join(hosts)))
    partner_seg = by_length[1]
    scored_copy = sorted(k for k in COF if k[0] == canonical_seg)[0]

    groups = collections.OrderedDict()
    groups["sub1"] = SEG[canonical_seg]
    groups["sub2"] = SEG[partner_seg]
    groups["dimer"] = sorted(SEG[canonical_seg] + SEG[partner_seg])
    groups["cofactor"] = sorted(COF[scored_copy])
    all_cof = sorted(i for v in COF.values() for i in v)
    groups["cofactors"] = all_cof
    groups["solute"] = sorted(set(groups["dimer"]) | set(all_cof))
    lost = sorted(i for (sg, ri, an), i in BY.items()
                  if sg == partner_seg and LOST_FROM <= ri <= LOST_TO
                  and res_of.get((sg, ri)) in AA_PLUS)
    if lost:
        groups["partner_lost_block"] = lost
    retained = sorted(i for (sg, ri, an), i in BY.items()
                      if sg == partner_seg and ri <= RETAINED_END
                      and res_of.get((sg, ri)) in AA_PLUS)
    if not retained:
        raise SystemExit("STOP: the partner chain %s carries no residue at or below "
                         "%d, so it is not the chain the spec describes"
                         % (partner_seg, RETAINED_END))
    groups["partner_retained_block"] = retained

    with open("iface.ndx", "w") as fh:
        for nm, ser in groups.items():
            fh.write("[ %s ]\n" % nm)
            for i in range(0, len(ser), 15):
                fh.write(" ".join(str(x) for x in ser[i:i + 15]) + "\n")
    print("  canonical subunit %s with %d residues, partner %s with %d, cofactor "
          "scored on %s%d" % (canonical_seg, lengths[canonical_seg], partner_seg,
                              lengths[partner_seg], scored_copy[0], scored_copy[1]))
    print("  index groups: %s" % ", ".join("%s (%d atoms)" % (k, len(v))
                                           for k, v in groups.items()))
    if not lost:
        print("  the partner carries no residue in %d to %d, which is the truncation "
              "itself, so that group is absent by construction" % (LOST_FROM, LOST_TO))
    return groups, prot, dict(canonical=canonical_seg, partner=partner_seg,
                              lengths=lengths, scored_copy=list(scored_copy),
                              has_lost_block=bool(lost))


def sasa(surface, out, label, output=None):
    args = ["sasa", "-s", "prod_01.tpr", "-f", "prod_fit.xtc", "-n", "iface.ndx",
            "-surface", surface, "-o", out, "-xvg", "none"]
    if output is not None:
        args += ["-output", output]
    gmx(*args, label=label, quiet=True)
    D = xvg(out, label)
    return D[:, 2] if output is not None else D[:, 1]


def measure(run, build, tpr, groups, prot, roles, natoms):
    cof = sasa('group "solute"', "sasa_cofactor.xvg", "cofactor exposure",
               output='group "cofactor"') * 100.0
    ab = sasa('group "dimer"', "sasa_dimer.xvg", "sasa dimer")
    a = sasa('group "sub1"', "sasa_sub1.xvg", "sasa subunit 1")
    b = sasa('group "sub2"', "sasa_sub2.xvg", "sasa subunit 2")
    n = min(len(cof), len(ab), len(a), len(b))
    if n < 10:
        raise SystemExit("STOP: the surface series hold %d frames between them" % n)
    buried = (a[:n] + b[:n] - ab[:n]) * 100.0 / 2.0
    cof = cof[:n]
    burn = int(n * BURN_FRACTION)

    gmx("mindist", "-s", tpr, "-f", "prod_fit.xtc", "-n", "iface.ndx", "-d", "0.5",
        "-od", "mindist_dimer.xvg", "-on", "numcont_dimer.xvg", "-xvg", "none",
        stdin_text="sub1\nsub2\n", label="mindist between the subunits", quiet=True)
    md_dimer = xvg("mindist_dimer.xvg", "mindist")[:, 1] * 10.0
    nc_dimer = xvg("numcont_dimer.xvg", "contact count")[:, 1]

    gmx("mindist", "-s", tpr, "-f", "prod_fit.xtc", "-n", "iface.ndx",
        "-od", "mindist_cofactor_partner.xvg", "-xvg", "none",
        stdin_text="cofactor\nsub2\n", label="cofactor to partner", quiet=True)
    md_cof = xvg("mindist_cofactor_partner.xvg", "mindist")[:, 1] * 10.0

    gmx("rms", "-s", tpr, "-f", "prod_fit.xtc", "-n", "iface.ndx", "-o", "rmsd.xvg",
        "-xvg", "none", stdin_text="dimer\ndimer\n", label="rmsd", quiet=True)
    rmsd = xvg("rmsd.xvg", "gmx rms")[burn:, 1] * 10.0

    def stats(v):
        v = np.asarray(v, dtype=float)
        return dict(mean=float(v.mean()), sd=float(v.std()),
                    p10=float(np.percentile(v, 10)), p90=float(np.percentile(v, 90)),
                    first=float(v[0]), last=float(v[-1]))

    t_ps = xvg("mindist_dimer.xvg", "mindist")[:, 0]
    sampled_ns = float((t_ps[-1] - t_ps[0]) / 1000.0)
    if sampled_ns <= 0:
        raise SystemExit("STOP: the trajectory spans %g ns, so its time axis is wrong"
                         % sampled_ns)
    out = dict(build=build, run=run, atoms=int(natoms), segments=prot, roles=roles,
               frames=int(n), burn=int(burn), ns=sampled_ns,
               ns_requested=float(NS_PER_BUILD),
               cofactor=COFACTOR,
               cofactor_sasa_A2=stats(cof[burn:]),
               cofactor_sasa_A2_full_series_mean=float(cof.mean()),
               buried_per_subunit_A2=stats(buried[burn:]),
               mindist_subunits_A=stats(md_dimer[burn:]),
               mindist_cofactor_partner_A=stats(md_cof[burn:]),
               contacts_0_5nm=stats(nc_dimer[burn:]),
               rmsd_A=stats(rmsd),
               retained_end=RETAINED_END, lost_block=[LOST_FROM, LOST_TO],
               written=datetime.datetime.now().isoformat(timespec="seconds"))
    json.dump(out, open(os.path.join(RESULTS, "%s_report.json" % build), "w"), indent=2)
    print("  %d frames over %.1f ns, %d burned in" % (n, sampled_ns, burn))
    print("  cofactor exposure in the canonical subunit %.1f +/- %.1f A2"
          % (out["cofactor_sasa_A2"]["mean"], out["cofactor_sasa_A2"]["sd"]))
    print("  buried interfacial area %.0f +/- %.0f A2 per subunit"
          % (out["buried_per_subunit_A2"]["mean"], out["buried_per_subunit_A2"]["sd"]))
    print("  inter-subunit contacts within 5 A %.0f +/- %.0f"
          % (out["contacts_0_5nm"]["mean"], out["contacts_0_5nm"]["sd"]))
    print("  shortest distance from the cofactor to the partner %.2f +/- %.2f A"
          % (out["mindist_cofactor_partner_A"]["mean"],
             out["mindist_cofactor_partner_A"]["sd"]))
    print("  backbone RMSD %.2f +/- %.2f A"
          % (out["rmsd_A"]["mean"], out["rmsd_A"]["sd"]))
    return out


def psf_disulfides(run):
    path = os.path.join(run, "step3_input.psf")
    if not os.path.isfile(path):
        raise SystemExit("STOP: no %s, so the disulfides cannot be verified" % path)
    lines = open(path, errors="replace").read().splitlines()
    i = next(k for k, l in enumerate(lines) if "!NATOM" in l)
    nat = int(lines[i].split()[0])
    sg = {}
    for l in lines[i + 1:i + 1 + nat]:
        f = l.split()
        if len(f) >= 5 and f[4] == "SG" and f[3] == "CYS":
            sg[int(f[0])] = (f[1], int(f[2]))
    j = next(k for k, l in enumerate(lines) if "!NBOND" in l)
    nb = int(lines[j].split()[0])
    tok = []
    for l in lines[j + 1:]:
        tok += [int(x) for x in l.split()]
        if len(tok) >= 2 * nb:
            break
    return sorted({tuple(sorted((sg[a], sg[b])))
                   for a, b in zip(tok[0::2], tok[1::2]) if a in sg and b in sg})


def xvg(path, what):
    must_exist(path, what)
    D = np.loadtxt(path, comments=["#", "@"])
    if D.ndim == 1:
        D = D.reshape(-1, len(D)) if D.size else D.reshape(0, 0)
    if not D.size:
        raise SystemExit("STOP: %s is empty, so %s produced no data" % (path, what))
    return D


def equilibrate(run):
    if not os.path.isfile("step4.0_minimization.tpr"):
        gmx("grompp", "-f", "step4.0_minimization.mdp", "-o",
            "step4.0_minimization.tpr", "-c", "step3_input.gro", "-r",
            "step3_input.gro", "-p", "topol.top", "-n", "index.ndx",
            "-maxwarn", MAXWARN, label="grompp minimisation")
        must_exist("step4.0_minimization.tpr", "grompp minimisation")
    if not os.path.isfile("step4.0_minimization.gro"):
        gmx("mdrun", "-v", "-deffnm", "step4.0_minimization", label="minimisation")
        must_exist("step4.0_minimization.gro", "minimisation")
    prev = "step4.0_minimization"
    stages = sorted(glob.glob("step4.?_equilibration.mdp"))
    if not stages:
        raise SystemExit("STOP: no step4.*_equilibration.mdp in %s. A CHARMM-GUI "
                         "solution build writes at least one, so the download is "
                         "incomplete." % run)
    for mdp in stages:
        name = mdp[:-4]
        if os.path.isfile(name + ".gro"):
            prev = name
            continue
        gmx("grompp", "-f", mdp, "-o", name + ".tpr", "-c", prev + ".gro", "-r",
            "step3_input.gro", "-p", "topol.top", "-n", "index.ndx",
            "-maxwarn", MAXWARN, label="grompp " + name)
        must_exist(name + ".tpr", "grompp " + name)
        gmx("mdrun", "-v", "-deffnm", name, label=name)
        must_exist(name + ".gro", name)
        prev = name
    return prev


def produce(run, build, eq_final):
    mdp = "step5_production.mdp"
    if not os.path.isfile(mdp):
        raise SystemExit("STOP: no %s in %s; the CHARMM-GUI download is incomplete"
                         % (mdp, run))
    dt = None
    for L in open(mdp):
        m = re.match(r"\s*dt\s*=\s*([\d.]+)", L)
        if m:
            dt = float(m.group(1))
    if dt is None:
        raise SystemExit("STOP: could not read dt from %s, so the chunk length is "
                         "unknown" % mdp)
    nsteps = int(round(NS_PER_CHUNK * 1000.0 / dt))
    n_chunks = int(math.ceil(float(NS_PER_BUILD) / NS_PER_CHUNK))
    per_chunk = want_frames(mdp, NS_PER_CHUNK)
    drive_run = RUN_DIR[build]
    prev = eq_final
    for i in range(1, n_chunks + 1):
        name = "prod_%02d" % i
        if os.path.isfile(name + ".gro"):
            got = xtc_frames(name + ".xtc")
            if got is not None and got >= per_chunk - 1:
                prev = name
                continue
        if budget_left() < 0.6:
            print("  wall budget nearly spent, stopping cleanly after %d of %d chunks"
                  % (i - 1, n_chunks))
            return prev, i - 1, n_chunks
        t0 = time.time()
        resume = os.path.isfile(name + ".cpt") and not os.path.isfile(name + ".gro")
        if not os.path.isfile(name + ".tpr"):
            gmx("grompp", "-f", mdp, "-o", name + ".tpr", "-c", prev + ".gro", "-p",
                "topol.top", "-n", "index.ndx", "-maxwarn", MAXWARN,
                label="grompp " + name)
        must_exist(name + ".tpr", "grompp " + name)
        extra = ("-cpi", name + ".cpt") if resume else ()
        if resume:
            print("  %s was interrupted, resuming from its checkpoint" % name)
        gmx("mdrun", "-v", "-deffnm", name, "-nsteps", nsteps, *extra, label=name)
        must_exist(name + ".gro", name)
        got = xtc_frames(name + ".xtc")
        if got is None or got < per_chunk - 1:
            raise SystemExit("STOP: %s wrote %s frames and should hold %d. The chunk "
                             "is incomplete, and concatenating it would give a "
                             "trajectory shorter than it reports."
                             % (name, got, per_chunk))
        print("  %s %s done in %.1f min, %d frames"
              % (build, name, (time.time() - t0) / 60.0, got))
        stage_out(os.getcwd(), drive_run,
                  [name + ".gro", name + ".cpt", name + ".xtc"])
        drive_wait(why="chunk %d" % i)
        progress(**{build: dict(chunks_done=i, chunks_total=n_chunks)})
        prev = name
    return prev, n_chunks, n_chunks


def concatenate(run, n_chunks, per_chunk):
    chunks = sorted(glob.glob("prod_[0-9][0-9].xtc"))
    if len(chunks) > n_chunks:
        raise SystemExit("STOP: %s holds %d chunk trajectories and the run asks for "
                         "%d: %s" % (os.getcwd(), len(chunks), n_chunks, chunks))
    if not chunks:
        raise SystemExit("STOP: %s holds no chunk trajectory" % os.getcwd())
    have_ns = len(chunks) * NS_PER_CHUNK
    counts = [xtc_frames(c) for c in chunks]
    short = [(c, n) for c, n in zip(chunks, counts)
             if n is None or n < per_chunk - 1]
    if short:
        raise SystemExit("STOP: %s hold fewer than the %d frames a chunk should "
                         "have. Concatenating them would produce a trajectory "
                         "shorter than it claims to be." % (short, per_chunk))
    for f in ("prod_all.xtc", "prod_nojump.xtc", "prod_fit.xtc"):
        if os.path.isfile(f):
            os.remove(f)
    offsets = "".join("%g\n" % (k * NS_PER_CHUNK * 1000.0) for k in range(len(chunks)))
    gmx("trjcat", "-f", *chunks, "-o", "prod_all.xtc", "-settime",
        stdin_text=offsets, label="trjcat")
    must_exist("prod_all.xtc", "trjcat")
    span = xtc_last_time("prod_all.xtc")
    if span is None or span < 0.9 * have_ns * 1000.0:
        raise SystemExit("STOP: prod_all.xtc ends at %s ps and the %d chunks that "
                         "finished should reach about %g ps. The time axis is wrong, "
                         "so every per-frame measurement downstream would be "
                         "mislabelled." % (span, len(chunks), have_ns * 1000.0))
    got = xtc_frames("prod_all.xtc")
    if got is None or got < sum(counts) - len(chunks):
        raise SystemExit("STOP: prod_all.xtc holds %s frames and the %d chunks hold "
                         "%d between them. The concatenation is incomplete."
                         % (got, len(chunks), sum(counts)))
    print("  concatenated %d chunks into %d frames, %d ns"
          % (len(chunks), got, have_ns))
    tpr = sorted(glob.glob("prod_[0-9][0-9].tpr"))[0]
    gmx("trjconv", "-s", tpr, "-f", "prod_all.xtc", "-o", "prod_nojump.xtc",
        *ndx_args(), "-pbc", "nojump", stdin_text="SOLU\n",
        label="trjconv nojump")
    must_exist("prod_nojump.xtc", "trjconv nojump")
    gmx("trjconv", "-s", tpr, "-f", "prod_nojump.xtc", "-o", "prod_fit.xtc",
        *ndx_args(), "-fit", "rot+trans", stdin_text="SOLU\nSOLU\n",
        label="trjconv fit")
    must_exist("prod_fit.xtc", "trjconv fit")
    gmx("trjconv", "-s", tpr, "-f", "prod_all.xtc", "-o", "prod_ref.pdb",
        *ndx_args(), "-dump", "0", stdin_text="SOLU\n", label="reference frame")
    must_exist("prod_ref.pdb", "reference frame")
    return tpr


def mdp_get(path, key):
    for l in open(path, errors="replace"):
        l = l.split(";")[0]
        if "=" in l and l.split("=")[0].strip().lower() == key.lower():
            return l.split("=", 1)[1].strip()
    return None


def mdp_set(path, key, value):
    out, hit = [], False
    for l in open(path, errors="replace"):
        body = l.split(";")[0]
        if "=" in body and body.split("=")[0].strip().lower() == key.lower():
            out.append("%-23s = %s\n" % (key, value)); hit = True
        else:
            out.append(l)
    if not hit:
        out.append("%-23s = %s\n" % (key, value))
    open(path, "w").writelines(out)

#@markdown ---
print("=" * 72)
print("PRE-FLIGHT: the two builds must differ only in the partner chain")
print("=" * 72)
ION = {"POT", "CLA", "SOD", "NA", "CL", "K", "KCL"}
pre = {}
for build in BUILD_ORDER:
    run = RUN_DIR[build]
    pdb = os.path.join(run, "step3_input.pdb")
    if not os.path.isfile(pdb):
        raise SystemExit("STOP: no %s. It is the gromacs/ folder from inside the "
                         "CHARMM-GUI download." % pdb)
    atoms = read_pdb_atoms_wide(pdb)
    res = {}
    for ser, seg, ri, name, an in atoms:
        res.setdefault((seg, ri), name)
    ions = collections.Counter(n for n in res.values() if n in ION)
    mdp = os.path.join(run, "step5_production.mdp")
    pre[build] = dict(atoms=len(atoms), ions=dict(ions), ss=len(psf_disulfides(run)),
                      prot=len(protein_segments_k(atoms)),
                      chains=sorted(len(v) for v in chain_residues(atoms).values()),
                      cofactors=len(cofactor_residues(atoms)),
                      ref_t=mdp_get(mdp, "ref-t") or mdp_get(mdp, "ref_t"),
                      dt=mdp_get(mdp, "dt"),
                      waters=sum(1 for n in res.values() if n in ("TIP3", "HOH", "SOL")))
    print("  %-10s %7d atoms, chains %s, %d %s, %d waters, ions %s, %d disulfides, "
          "ref_t %s" % (build, pre[build]["atoms"], pre[build]["chains"],
                        pre[build]["cofactors"], COFACTOR, pre[build]["waters"],
                        dict(ions) or "none", pre[build]["ss"], pre[build]["ref_t"]))

for build in BUILD_ORDER:
    if pre[build]["chains"] != sorted(CHAIN_LENGTHS[build]):
        raise SystemExit("STOP: %s carries chains of %s residues and the spec says "
                         "%s" % (build, pre[build]["chains"],
                                 sorted(CHAIN_LENGTHS[build])))
    if pre[build]["cofactors"] != COFACTOR_COPIES[build]:
        raise SystemExit("STOP: %s carries %d copies of %s and the spec says %d"
                         % (build, pre[build]["cofactors"], COFACTOR,
                            COFACTOR_COPIES[build]))
if len({pre[b]["ss"] for b in BUILD_ORDER}) != 1:
    raise SystemExit("STOP: the builds carry different numbers of disulfides, %s. The "
                     "two share their canonical chain, so a difference means CHARMM-GUI "
                     "built a bond in one that it did not build in the other."
                     % {b: pre[b]["ss"] for b in BUILD_ORDER})
if len({pre[b]["prot"] for b in BUILD_ORDER}) != 1:
    raise SystemExit("STOP: the builds disagree on the number of protein segments: %s"
                     % {b: pre[b]["prot"] for b in BUILD_ORDER})
for key in ("ref_t", "dt"):
    vals = {pre[b][key] for b in BUILD_ORDER}
    if len(vals) != 1 or None in vals:
        raise SystemExit("STOP: the builds disagree on %s: %s. The comparison needs "
                         "the same conditions on both."
                         % (key, {b: pre[b][key] for b in BUILD_ORDER}))
TEMPERATURE = pre[BUILD_ORDER[0]]["ref_t"]
got_t = [float(x) for x in str(TEMPERATURE).replace(",", " ").split()]
if not got_t or max(abs(v - REF_T_WANT) for v in got_t) > 0.1:
    raise SystemExit("STOP: CHARMM-GUI built these systems with ref_t %s and this "
                     "series runs at %g K (%s). A run at another temperature is not "
                     "comparable to the rest of the line, and the two builds matching "
                     "each other does not make either of them right. Set the "
                     "temperature on CHARMM-GUI's dynamics page and download again, or "
                     "change ref_t_K in md/inputs/build_spec.json if the series "
                     "temperature really has moved."
                     % (TEMPERATURE, REF_T_WANT, REF_T_SOURCE))
lost_residues = LOST_TO - RETAINED_END
spread = abs(pre["canonical"]["atoms"] - pre["mixed"]["atoms"])
allowance = 60 * lost_residues
if spread > allowance:
    raise SystemExit("STOP: the builds differ by %d atoms and the truncation removes "
                     "%d residues, which cannot account for more than about %d atoms "
                     "including their water. Rebuild both with the same edge distance "
                     "and the same salt." % (spread, lost_residues, allowance))
print("  chain lengths and cofactor copies agree with the spec in both builds")
print("  thermostat agrees across builds and with the spec: ref_t %s K" % TEMPERATURE)
print("  atom spread %d, within the %d the truncation allows" % (spread, allowance))
if len({tuple(sorted(pre[b]["ions"].items())) for b in BUILD_ORDER}) != 1:
    print("  note: the ion counts differ, %s. The two systems carry different net "
          "charge because one chain is shorter, so this is expected and is recorded "
          "rather than refused." % {b: pre[b]["ions"] for b in BUILD_ORDER})

for build in BUILD_ORDER:
    mdp = os.path.join(RUN_DIR[build], "step5_production.mdp")
    dt = float(mdp_get(mdp, "dt"))
    want = int(round(TOTAL_NS * 1000.0 / dt / TARGET_FRAMES))
    want = max(100, want - want % 100)
    have = mdp_get(mdp, "nstxout-compressed")
    if have is None or int(have) != want:
        mdp_set(mdp, "nstxout-compressed", want)
        print("  %-10s nstxout-compressed %s -> %d  (%d frames over the planned %d ns, "
              "%d over the %d ns asked for now)"
              % (build, have, want, TOTAL_NS * 1000.0 / dt / want, TOTAL_NS,
                 NS_PER_BUILD * 1000.0 / dt / want, NS_PER_BUILD))
    else:
        print("  %-10s nstxout-compressed already %d" % (build, want))
progress(preflight="passed", ref_t=TEMPERATURE, atom_spread=spread,
         ions=pre[BUILD_ORDER[0]]["ions"])
print("\npre-flight passed, starting the run\n")

incomplete = []
for build in [BUILD]:
    run = RUN_DIR[build]
    rep = os.path.join(RESULTS, "%s_report.json" % build)
    print("\n" + "=" * 72)
    print("BUILD %s   %s" % (build.upper(), run))
    print("=" * 72)
    if os.path.isfile(rep) and json.load(open(rep)).get("ns", 0) >= 0.95 * NS_PER_BUILD:
        print("  already finished at %.1f ns" % json.load(open(rep))["ns"])
        continue
    required = ["step3_input.gro", "step3_input.pdb", "step3_input.psf", "topol.top",
                "step5_production.mdp", "index.ndx"]
    miss = [f for f in required if not os.path.isfile(os.path.join(run, f))]
    if miss:
        raise SystemExit("STOP: %s is missing %s. It should be the gromacs/ folder "
                         "from inside the CHARMM-GUI download." % (run, miss))
    if not globals().get("HAVE_GPU", False):
        raise SystemExit("STOP: build %s needs a GPU. Equilibration and production "
                         "both ask mdrun to offload, and section 2 found no usable "
                         "GPU, so this would fail hours from now with a message about "
                         "offload instead of about the runtime. Runtime > Change "
                         "runtime type > GPU, then run from the top." % build)
    natoms, prot = check_identity(run, build)
    n_chunks = int(math.ceil(float(NS_PER_BUILD) / NS_PER_CHUNK))
    per_chunk = want_frames(os.path.join(run, "step5_production.mdp"), NS_PER_CHUNK)
    work = os.path.join("/content/work", build)
    print("  a chunk should hold %d frames" % per_chunk)
    stage_in(run, work, n_chunks, per_chunk)
    os.chdir(work)
    eq = equilibrate(work)
    progress(**{build: dict(stage="equilibrated")})
    prev, done, total = produce(work, build, eq)
    if done == 0:
        incomplete.append((build, 0, total))
        continue
    tpr = concatenate(work, done, per_chunk)
    groups, segs, roles = interface_groups(build)
    measure(run, build, tpr, groups, segs, roles, natoms)
    stage_out(work, run, ["prod_fit.xtc", "prod_ref.pdb", os.path.basename(tpr),
                          "iface.ndx", "sasa_cofactor.xvg", "sasa_dimer.xvg",
                          "sasa_sub1.xvg", "sasa_sub2.xvg", "mindist_dimer.xvg",
                          "numcont_dimer.xvg", "mindist_cofactor_partner.xvg",
                          "rmsd.xvg"])
    drive_wait(why="the analysis files")
    print("  copied the fitted trajectory and its analysis files to Drive")
    if done < total:
        incomplete.append((build, done, total))
    progress(**{build: dict(stage="measured", chunks_done=done, chunks_total=total)})

os.chdir("/content")
drive_verify()
if incomplete:
    for b, d, t in incomplete:
        print("%s reached %d of %d chunks" % (b, d, t))
    print("\nThe wall budget ran out before %s reached %d ns. Nothing is lost: the "
          "chunks that finished are measured and reported, and opening this notebook "
          "again and pressing Run all continues from the last one. The cached GROMACS "
          "means the next session starts in about a minute." % (BUILD, NS_PER_BUILD))
    print("Part 3 scores what is there: it can stop the series on a short run, and it "
          "withholds a pass until the full %d ns." % NS_PER_BUILD)
print("\n%s finished. Run the other build in its own notebook, then open "
      "kynu_MD_3_compare.ipynb, which applies the criterion." % BUILD.upper())
